# Tests in Dimensions Less Than Six
DAJones 
March 2026

With the time testing accomplished in 6D, here we time the lesser dimensions.
The 6D algorithms work in lesser dimensions without modification,
but we want to find out the savings and penalties for casing the dimensions
(like A&D) suggested.

We ran into a problem with the Hitzer algorithms for 4D and 5D.


In [1]:
import timeit
import numpy as np
import clifford as cf
from jones_6D_inverse import jones_6D_inverse, jones_inverse
from acus_6D_inverse import acus_6D_inverse, acus_inverse

import warnings
from numba import NumbaDeprecationWarning
from numba import NumbaPendingDeprecationWarning
# Silence the noise
warnings.simplefilter('ignore', category=NumbaDeprecationWarning)
warnings.simplefilter('ignore', category=NumbaPendingDeprecationWarning)

# Results storage
results = {}

print(f"{'Dim':<5} | {'Algorithm':<18} | {'Avg Time (usec)':<15}")
print("-" * 45)

for d in range(1, 6):
    # 1. Setup layout for dimension d
    layout, _ = cf.Cl(d, 0)
    num_coeffs = 2**d
    
    # 2. Generate a random dense multivector
    A = layout.MultiVector(np.random.randn(num_coeffs))
    
    # 3. Define local wrappers for timing
    # These capture the current 'A' in the loop
    benchmarks = {
        "Shirokov":    lambda: A.shirokov_inverse(),
        "Perwass":     lambda: A.leftLaInv(),
        "Hitzer":      lambda: A.hitzer_inverse(),
        "Jones 6D":    lambda: jones_6D_inverse(A),
        "Jones cased": lambda: jones_inverse(A),
        "Acus 6D":     lambda: acus_6D_inverse(A),
        "Acus cased":  lambda: acus_inverse(A)
    }

    n_it = 2000 # Higher iterations for lower dimensions to stabilize stats
    
    for name, func in benchmarks.items():
        try:
            # Warm up for JIT/Numba
            func()
            
            # Time execution
            t = timeit.timeit(func, number=n_it)
            avg_t = 1000000 * t / n_it
            
            print(f"{d:<5} | {name:<18} | {avg_t:.3f}")
            
        except Exception:
            # Skip algorithms not supported in specific dimensions
            continue
    print("-" * 45)

Dim   | Algorithm          | Avg Time (usec)
---------------------------------------------
1     | Shirokov           | 2.982
1     | Perwass            | 2.824
1     | Hitzer             | 2.997
1     | Jones 6D           | 31.121
1     | Jones cased        | 27.262
1     | Acus 6D            | 73.006
1     | Acus cased         | 23.422
---------------------------------------------
2     | Shirokov           | 3.106
2     | Perwass            | 3.156
2     | Hitzer             | 3.098
2     | Jones 6D           | 31.942
2     | Jones cased        | 27.747
2     | Acus 6D            | 71.718
2     | Acus cased         | 24.031
---------------------------------------------
3     | Shirokov           | 3.849
3     | Perwass            | 4.570
3     | Hitzer             | 3.706
3     | Jones 6D           | 32.402
3     | Jones cased        | 28.198
3     | Acus 6D            | 74.709
3     | Acus cased         | 66.996
---------------------------------------------
4     | Shirokov        